# RunGap Corridor Environment Visualization

This notebook demonstrates the RunGap corridor environment with:
1. Arena rendering (overview of the gap corridor with **mesh platforms** for warp texture support)
2. Zero-action rollout with reward/position plots
3. Video from `close_profile-rodent` camera
4. **JAX-native egocentric vision** using the warp GPU ray-tracer (`JaxVisionRenderer`) — with textured mesh platforms
5. Concatenated video with vision overlay in upper-left corner

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
import jax
import jax.numpy as jp
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np
from tqdm import tqdm

from vnl_playground.tasks.rodent import run_gap

## 1. Initialize Environment and Render Arena

In [ ]:
from vnl_playground.tasks.rodent.run_gap import default_config

# Use mesh platforms so warp renderer can display textures on platforms
# (warp's sample_texture() only supports PLANE and MESH geom types, not BOX)
cfg = default_config()
cfg.use_mesh_platforms = True
env = run_gap.RunGap(config=cfg)
mj_model = env.mj_model
fps = int(1.0 / env.dt)

# Compute corridor end position (env no longer stores _corridor_end_x)
if hasattr(env, "_reference_positions"):
    # randomize_gaps=True: use max-spacing reference positions
    corridor_end_x = float(env._reference_positions[-1]) + env._platform_half_length
else:
    # randomize_gaps=False: use static trailing edge of last platform
    corridor_end_x = float(env._static_platform_trailing_edges[-1])

print(f"Action size: {env.action_size}")
print(f"Obs size: {env.observation_size}")
print(f"FPS: {fps}")
print(f"Corridor end x: {corridor_end_x:.2f}m")
print(f"Mesh platforms: nmesh={mj_model.nmesh} (texture-capable in warp)")
print(f"Cameras: {[mj_model.camera(i).name for i in range(mj_model.ncam)]}")

In [ ]:
# Render a few static views of the arena
mj_data = mujoco.MjData(mj_model)
mujoco.mj_forward(mj_model, mj_data)

renderer = mujoco.Renderer(mj_model, height=480, width=640)

# Overview camera positions along the corridor
cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE
cam.distance = 3.0
cam.elevation = -40

positions = [
    0.0,
    corridor_end_x / 3,
    2 * corridor_end_x / 3,
    corridor_end_x,
]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, x_pos in zip(axes, positions):
    cam.azimuth = 90
    cam.lookat[:] = [x_pos, 0.0, 0.0]
    renderer.update_scene(mj_data, camera=cam)
    ax.imshow(renderer.render())
    ax.set_title(f"x = {x_pos:.1f}m")
    ax.axis("off")
plt.suptitle("Arena Overview - Side Views Along Corridor", fontsize=14)
plt.tight_layout()
plt.show()

# Top-down view of full corridor
cam.distance = 8.0
cam.elevation = -89
cam.azimuth = 0
cam.lookat[:] = [corridor_end_x / 2, 0.0, 0.0]
renderer.update_scene(mj_data, camera=cam)
fig, ax = plt.subplots(1, 1, figsize=(16, 3))
ax.imshow(renderer.render())
ax.set_title("Top-Down View of Corridor")
ax.axis("off")
plt.tight_layout()
plt.show()

renderer.close()

## 1b. Outdoor Natural Aesthetic Comparison

Compare the default (grey/dark) arena with the enriched outdoor natural aesthetic
(grass-textured platforms, blue sky skybox) inspired by dm_control's `outdoor_natural` style.

In [ ]:
# Initialize outdoor natural environment (also with mesh platforms for warp textures)
outdoor_config = default_config()
outdoor_config.aesthetic = "outdoor_natural"
outdoor_config.use_mesh_platforms = True
env_outdoor = run_gap.RunGap(config=outdoor_config)
mj_model_outdoor = env_outdoor.mj_model

print(
    f"Outdoor env - Textures: {mj_model_outdoor.ntex}, Materials: {mj_model_outdoor.nmat}, "
    f"Meshes: {mj_model_outdoor.nmesh}"
)

# Compute corridor end for outdoor env
if hasattr(env_outdoor, "_reference_positions"):
    corridor_end_x_outdoor = (
        float(env_outdoor._reference_positions[-1]) + env_outdoor._platform_half_length
    )
else:
    corridor_end_x_outdoor = float(env_outdoor._static_platform_trailing_edges[-1])

In [ ]:
# Side-by-side comparison: Default vs Outdoor Natural
mj_data_default = mujoco.MjData(mj_model)
mj_data_outdoor = mujoco.MjData(mj_model_outdoor)
mujoco.mj_forward(mj_model, mj_data_default)
mujoco.mj_forward(mj_model_outdoor, mj_data_outdoor)

renderer_default = mujoco.Renderer(mj_model, height=480, width=640)
renderer_outdoor = mujoco.Renderer(mj_model_outdoor, height=480, width=640)

# Compare at multiple positions along the corridor
positions_cmp = [0.0, corridor_end_x / 3, 2 * corridor_end_x / 3]

fig, axes = plt.subplots(2, len(positions_cmp), figsize=(5 * len(positions_cmp), 8))

for col, x_pos in enumerate(positions_cmp):
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.distance = 3.0
    cam.elevation = -40
    cam.azimuth = 90
    cam.lookat[:] = [x_pos, 0.0, 0.0]

    # Default aesthetic
    renderer_default.update_scene(mj_data_default, camera=cam)
    axes[0, col].imshow(renderer_default.render())
    axes[0, col].set_title(f"x = {x_pos:.1f}m")
    axes[0, col].axis("off")

    # Outdoor natural aesthetic
    renderer_outdoor.update_scene(mj_data_outdoor, camera=cam)
    axes[1, col].imshow(renderer_outdoor.render())
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Default\n(grey/dark)", fontsize=12)
axes[1, 0].set_ylabel("Outdoor Natural\n(grass/sky)", fontsize=12)
plt.suptitle("Arena Aesthetic Comparison: Default vs Outdoor Natural", fontsize=14)
plt.tight_layout()
plt.show()

# Top-down comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

cam_top = mujoco.MjvCamera()
cam_top.type = mujoco.mjtCamera.mjCAMERA_FREE
cam_top.distance = 8.0
cam_top.elevation = -89
cam_top.azimuth = 0
cam_top.lookat[:] = [corridor_end_x / 2, 0.0, 0.0]

renderer_default.update_scene(mj_data_default, camera=cam_top)
axes[0].imshow(renderer_default.render())
axes[0].set_title("Default - Top Down")
axes[0].axis("off")

renderer_outdoor.update_scene(mj_data_outdoor, camera=cam_top)
axes[1].imshow(renderer_outdoor.render())
axes[1].set_title("Outdoor Natural - Top Down")
axes[1].axis("off")

plt.suptitle("Top-Down View Comparison", fontsize=14)
plt.tight_layout()
plt.show()

renderer_default.close()
renderer_outdoor.close()

## 1c. Why Mesh Platforms? Box vs Mesh in the Warp Renderer

The main environment (Section 1) uses `use_mesh_platforms=True`. Here's why:

The warp GPU ray-tracer's `sample_texture()` function only supports **PLANE** and
**MESH** geom types — it **cannot** sample textures on BOX geoms. With standard box
geom platforms, the warp renderer shows them with flat/untextured shading even when a
textured material is assigned.

Setting `use_mesh_platforms=True` replaces each platform's box geom with an equivalent
mesh geom that has proper UV texture coordinates, enabling the warp renderer to display
textures. The cells below compare box vs mesh to illustrate this difference.

In [ ]:
# Initialize a box-platform env for comparison (the main `env` already uses mesh)
config_box = default_config()
config_box.use_mesh_platforms = False
env_box = run_gap.RunGap(config=config_box)

# The main env (mesh mode) for comparison
env_mesh = env  # already initialized with use_mesh_platforms=True

# Compare model statistics
print("=== Box Platforms (use_mesh_platforms=False) ===")
print(f"  ngeom: {env_box.mj_model.ngeom}")
print(f"  nmesh: {env_box.mj_model.nmesh}")
print(f"  ntex:  {env_box.mj_model.ntex}")
print(f"  nmat:  {env_box.mj_model.nmat}")

print("\n=== Mesh Platforms (use_mesh_platforms=True) — main env ===")
print(f"  ngeom: {env_mesh.mj_model.ngeom}")
print(f"  nmesh: {env_mesh.mj_model.nmesh}")
print(f"  ntex:  {env_mesh.mj_model.ntex}")
print(f"  nmat:  {env_mesh.mj_model.nmat}")

print(
    f"\nMesh mode adds {env_mesh.mj_model.nmesh - env_box.mj_model.nmesh} meshes "
    f"for platform geoms (UV coords enable warp texture sampling)."
)

In [ ]:
# CPU renderer side-by-side: Box vs Mesh platforms
# The MuJoCo CPU renderer (OpenGL-based) can texture both box and mesh geoms,
# so both should look similar here. The difference only shows in the warp renderer.

mj_data_box = mujoco.MjData(env_box.mj_model)
mj_data_mesh = mujoco.MjData(mj_model)  # main env uses mesh platforms
mujoco.mj_forward(env_box.mj_model, mj_data_box)
mujoco.mj_forward(mj_model, mj_data_mesh)

renderer_box = mujoco.Renderer(env_box.mj_model, height=480, width=640)
renderer_mesh = mujoco.Renderer(mj_model, height=480, width=640)

positions_cmp = [0.0, corridor_end_x / 3, 2 * corridor_end_x / 3]

fig, axes = plt.subplots(2, len(positions_cmp), figsize=(5 * len(positions_cmp), 8))

for col, x_pos in enumerate(positions_cmp):
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.distance = 3.0
    cam.elevation = -40
    cam.azimuth = 90
    cam.lookat[:] = [x_pos, 0.0, 0.0]

    # Box platforms (row 0)
    renderer_box.update_scene(mj_data_box, camera=cam)
    axes[0, col].imshow(renderer_box.render())
    axes[0, col].set_title(f"x = {x_pos:.1f}m")
    axes[0, col].axis("off")

    # Mesh platforms (row 1)
    renderer_mesh.update_scene(mj_data_mesh, camera=cam)
    axes[1, col].imshow(renderer_mesh.render())
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Box Platforms\n(default)", fontsize=12)
axes[1, 0].set_ylabel("Mesh Platforms\n(use_mesh_platforms=True)", fontsize=12)
plt.suptitle(
    "CPU Renderer: Box vs Mesh Platforms (both textured by CPU renderer)", fontsize=14
)
plt.tight_layout()
plt.show()

renderer_box.close()
renderer_mesh.close()

In [ ]:
# Warp GPU renderer comparison: Box geom vs Mesh geom texture rendering
#
# This is the key validation: warp's sample_texture() only supports PLANE and
# MESH geom types. Box geoms get flat shading even when a texture is assigned.
# Mesh platforms produce proper textured rendering in the warp ray-tracer.

from vnl_playground.tasks.rodent.vision_jax import JaxVisionRenderer
from mujoco.mjx.warp.types import DATA_NON_VMAP

NWORLD = 1
WIDTH, HEIGHT = 64, 64


def add_batch_dim(data):
    """Add a leading nworld=1 dimension to mjx.Data, skipping non-vmap fields."""

    def _maybe_expand(path, x):
        parts = [p.name for p in path if hasattr(p, "name") and p.name != "_impl"]
        attr = "__".join(parts)
        if attr in DATA_NON_VMAP:
            return x
        return x[None, ...]

    return jax.tree.map_with_path(_maybe_expand, data)


# --- Run short zero-action rollouts on both envs to get mjx.Data states ---
n_rollout = 30

# Box platform rollout
rng_box = jax.random.PRNGKey(42)
state_box = jax.jit(env_box.reset)(rng_box)
step_fn_box = jax.jit(env_box.step)
rollout_box = []
for i in tqdm(range(n_rollout), desc="Box platform rollout"):
    action = jp.zeros(env_box.action_size)
    state_box = step_fn_box(state_box, action)
    rollout_box.append(state_box)

# Mesh platform rollout (main env)
rng_mesh = jax.random.PRNGKey(42)
state_mesh = jax.jit(env.reset)(rng_mesh)
step_fn_mesh = jax.jit(env.step)
rollout_mesh = []
for i in tqdm(range(n_rollout), desc="Mesh platform rollout"):
    action = jp.zeros(env.action_size)
    state_mesh = step_fn_mesh(state_mesh, action)
    rollout_mesh.append(state_mesh)

print(
    f"Rollouts complete: {len(rollout_box)} box steps, {len(rollout_mesh)} mesh steps"
)

# --- Create JaxVisionRenderers for both ---
vision_renderer_box = JaxVisionRenderer(
    mj_model=env_box.mj_model,
    mjx_model=env_box.mjx_model,
    nworld=NWORLD,
    width=WIDTH,
    height=HEIGHT,
    grayscale=False,
    render_depth=False,
    use_shadows=True,
    use_textures=True,
    camera_name="egocentric-rodent",
)

vision_renderer_mesh_cmp = JaxVisionRenderer(
    mj_model=mj_model,
    mjx_model=env.mjx_model,
    nworld=NWORLD,
    width=WIDTH,
    height=HEIGHT,
    grayscale=False,
    render_depth=False,
    use_shadows=True,
    use_textures=True,
    camera_name="egocentric-rodent",
)

print(f"Warp renderers initialized: {WIDTH}x{HEIGHT} RGB")


@jax.jit
def render_warp_box(data):
    batched = add_batch_dim(data)
    return vision_renderer_box.render(batched)[0]


@jax.jit
def render_warp_mesh_cmp(data):
    batched = add_batch_dim(data)
    return vision_renderer_mesh_cmp.render(batched)[0]


# Render selected frames from both rollouts
sample_steps = np.linspace(0, len(rollout_box) - 1, 6, dtype=int)

warp_box_frames = []
warp_mesh_frames = []
for idx in tqdm(sample_steps, desc="Warp rendering"):
    warp_box_frames.append(np.array(render_warp_box(rollout_box[idx].data)))
    warp_mesh_frames.append(np.array(render_warp_mesh_cmp(rollout_mesh[idx].data)))

# Plot side-by-side
fig, axes = plt.subplots(2, len(sample_steps), figsize=(3 * len(sample_steps), 6))
for col, idx in enumerate(sample_steps):
    axes[0, col].imshow(warp_box_frames[col])
    axes[0, col].set_title(f"Step {idx}", fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(warp_mesh_frames[col])
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Box Geom\n(no texture in warp)", fontsize=11)
axes[1, 0].set_ylabel("Mesh Geom\n(textured in warp)", fontsize=11)
plt.suptitle(
    "Warp Texture Rendering: Box Geom (flat color) vs Mesh Geom (textured)",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## 2. Zero-Action Rollout

In [ ]:
rng = jax.random.PRNGKey(0)
state = jax.jit(env.reset)(rng)
step_fn = jax.jit(env.step)

n_steps = 100
rollout_states = []
qposes = [np.array(state.data.qpos)]
rewards = []
positions = []

torso = state.data.bind(env.mjx_model, env._spec.body("torso-rodent"))
positions.append(np.array(torso.xpos))

for i in tqdm(range(n_steps), desc="Zero-action rollout"):
    action = jp.zeros(env.action_size)
    state = step_fn(state, action)
    rollout_states.append(state)
    qposes.append(np.array(state.data.qpos))
    rewards.append(float(state.reward))
    torso = state.data.bind(env.mjx_model, env._spec.body("torso-rodent"))
    positions.append(np.array(torso.xpos))
    if state.done > 0.5:
        break

qposes = np.array(qposes)
rewards = np.array(rewards)
positions = np.array(positions)

print(f"Steps: {len(rewards)}")
print(f"Mean reward: {rewards.mean():.4f}")
print(f"Final x-pos: {positions[-1, 0]:.4f}m")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(rewards)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Reward")
axes[0].set_title("Reward over time")

axes[1].plot(positions[:, 0], label="X")
axes[1].plot(positions[:, 1], label="Y")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Position (m)")
axes[1].set_title("XY Position")
axes[1].legend()

axes[2].plot(positions[:, 2])
axes[2].axhline(y=-0.05, color="r", linestyle="--", label="Fall threshold")
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Z (m)")
axes[2].set_title("Height over time")
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Video from close_profile-rodent Camera

In [ ]:
mj_data = mujoco.MjData(mj_model)
renderer = mujoco.Renderer(mj_model, height=480, width=640)

render_every = 2
frames_profile = []
for qpos in tqdm(qposes[::render_every], desc="Rendering close_profile"):
    mj_data.qpos = qpos
    mujoco.mj_forward(mj_model, mj_data)
    renderer.update_scene(mj_data, camera="close_profile-rodent")
    frames_profile.append(renderer.render().copy())

renderer.close()
print(f"Rendered {len(frames_profile)} frames")
media.show_video(frames_profile, fps=fps // render_every)

## 4. JAX-Native Egocentric Vision (Warp GPU Ray-Tracer)

This uses the `JaxVisionRenderer` from `vision_jax.py`, which wraps the
`create_mjx_render_fn` factory from the MJX warp backend. The renderer:

- Is **JAX-traceable** — works inside `jax.jit` and `jax.lax.scan`
- Captures static warp state (Model, Data, RenderContext) in a closure
- Only dynamic kinematic arrays flow through JAX
- Produces float32 images in [0, 1] (grayscale or RGB)

Because the environment uses **mesh platforms** (`use_mesh_platforms=True`), the warp
renderer can sample textures on the platforms. Without this, box geoms would render as
flat colors (see Section 1c for comparison).

This is the same renderer used during training via `VisionRenderWrapper`.

In [ ]:
from vnl_playground.tasks.rodent.vision_jax import JaxVisionRenderer

# Create renderer for batched MJX rollout
# nworld=1 since our rollout is single-world
NWORLD = 1
WIDTH, HEIGHT = 64, 64

vision_renderer = JaxVisionRenderer(
    mj_model=mj_model,
    mjx_model=env.mjx_model,
    nworld=NWORLD,
    width=WIDTH,
    height=HEIGHT,
    grayscale=False,  # RGB for visualization
    render_depth=True,
    use_shadows=True,
    use_textures=True,
    camera_name="egocentric-rodent",
)

print(f"JaxVisionRenderer initialized: {NWORLD} world(s), {WIDTH}x{HEIGHT}")
print(f"Vision shape: {vision_renderer.vision_shape}")

In [ ]:
# Render vision from the Section 2 rollout states using JaxVisionRenderer.
#
# The rollout states already contain warp-backed mjx.Data with all kinematic
# arrays (geom_xpos, cam_xpos, etc.) populated from the physics step.
# We just need to add a batch dimension (nworld=1) for the renderer.
#
# IMPORTANT: We must skip DATA_NON_VMAP fields (contact, efc, etc.) when adding
# the batch dimension -- the warp FFI layer expects them unbatched.

n_vision_steps = min(500, len(rollout_states))

from mujoco.mjx.warp.types import DATA_NON_VMAP


def add_batch_dim(data):
    """Add a leading nworld=1 dimension to mjx.Data, skipping non-vmap fields."""

    def _maybe_expand(path, x):
        parts = [p.name for p in path if hasattr(p, "name") and p.name != "_impl"]
        attr = "__".join(parts)
        if attr in DATA_NON_VMAP:
            return x
        return x[None, ...]

    return jax.tree.map_with_path(_maybe_expand, data)


@jax.jit
def render_from_data(data):
    """Render from mjx.Data (already has kinematic arrays from physics step).

    Returns (rgb, depth) if render_depth=True, else just rgb.
    All outputs have the batch dim stripped: (H, W, C) not (1, H, W, C).
    """
    batched_data = add_batch_dim(data)
    result = vision_renderer.render(batched_data)
    if isinstance(result, tuple):
        # render_depth=True: (images, depth) — strip batch dim from both
        return result[0][0], result[1][0]
    return result[0]  # strip batch dim


print(
    f"Rendering {n_vision_steps // render_every} vision frames via JaxVisionRenderer..."
)

# Render vision for each saved state from the Section 2 rollout
vision_images = []
depth_images = []
for state in tqdm(
    rollout_states[:n_vision_steps:render_every], desc="JAX vision render"
):
    result = render_from_data(state.data)
    if isinstance(result, tuple):
        vision_images.append(np.array(result[0]))  # (H, W, C)
        depth_images.append(np.array(result[1]))
    else:
        vision_images.append(np.array(result))  # (H, W, C)

print(f"Collected {len(vision_images)} vision frames")
print(f"Frame shape: {vision_images[0].shape}, dtype: {vision_images[0].dtype}")
print(f"Value range: [{vision_images[0].min():.3f}, {vision_images[0].max():.3f}]")
if depth_images:
    print(
        f"Depth shape: {depth_images[0].shape}, range: [{depth_images[0].min():.3f}, {depth_images[0].max():.3f}]"
    )

In [ ]:
# Show sample vision frames
sample_indices = np.linspace(0, len(vision_images) - 1, 10, dtype=int)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, idx in zip(axes.flat, sample_indices):
    ax.imshow(vision_images[idx])
    ax.set_title(f"Step {idx * render_every}")
    ax.axis("off")
plt.suptitle(
    f"Egocentric Camera ({WIDTH}x{HEIGHT}) — JAX-native Warp Renderer", fontsize=14
)
plt.tight_layout()
plt.show()

In [ ]:
# Compare: MuJoCo CPU renderer vs JAX-native Warp renderer (egocentric camera)
mj_data_cmp = mujoco.MjData(mj_model)
renderer_cmp = mujoco.Renderer(mj_model, height=HEIGHT, width=WIDTH)

compare_indices = [0, len(rollout_states) // 4, len(rollout_states) // 2]
fig, axes = plt.subplots(2, len(compare_indices), figsize=(4 * len(compare_indices), 8))

for col, idx in enumerate(compare_indices):
    state = rollout_states[idx]
    mj_data_cmp.qpos = np.array(state.data.qpos)
    mujoco.mj_forward(mj_model, mj_data_cmp)

    # Row 0: MuJoCo CPU renderer (egocentric camera)
    renderer_cmp.update_scene(mj_data_cmp, camera="egocentric-rodent")
    axes[0, col].imshow(renderer_cmp.render().copy())
    axes[0, col].set_title(f"Step {idx}")
    axes[0, col].axis("off")

    # Row 1: JAX-native Warp renderer (same state)
    result = render_from_data(state.data)
    warp_img = result[0] if isinstance(result, tuple) else result
    axes[1, col].imshow(np.array(warp_img))
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("MuJoCo CPU\nRenderer", fontsize=11)
axes[1, 0].set_ylabel("Warp GPU\n(JAX-native)", fontsize=11)
plt.suptitle("Egocentric Camera Comparison: MuJoCo CPU vs Warp GPU", fontsize=14)
plt.tight_layout()
plt.show()

renderer_cmp.close()

In [ ]:
# Show vision as video
vision_uint8 = [np.clip(f * 255, 0, 255).astype(np.uint8) for f in vision_images]
media.show_video(
    vision_uint8,
    fps=fps // render_every,
    title=f"Egocentric Vision ({WIDTH}x{HEIGHT}, JAX-native Warp Renderer)",
)

## 4b. Outdoor Natural Egocentric Vision

Compare the default (grey/dark) and outdoor natural (grass/sky) aesthetics as seen
through the JAX-native warp GPU ray-tracer at the same resolution used during training.

In [ ]:
# Create vision renderer for the outdoor natural environment
vision_renderer_outdoor = JaxVisionRenderer(
    mj_model=mj_model_outdoor,
    mjx_model=env_outdoor.mjx_model,
    nworld=NWORLD,
    width=WIDTH,
    height=HEIGHT,
    grayscale=False,  # RGB for visualization
    render_depth=True,
    use_shadows=True,
    use_textures=True,
    camera_name="egocentric-rodent",
)
print(f"Outdoor JaxVisionRenderer initialized: {NWORLD} world(s), {WIDTH}x{HEIGHT}")

# Run a zero-action rollout on the outdoor env (need fresh mjx.Data for this model)
rng_outdoor = jax.random.PRNGKey(0)
state_outdoor = jax.jit(env_outdoor.reset)(rng_outdoor)
step_fn_outdoor = jax.jit(env_outdoor.step)

n_steps_outdoor = 100
rollout_states_outdoor = []
for i in tqdm(range(n_steps_outdoor), desc="Outdoor zero-action rollout"):
    action = jp.zeros(env_outdoor.action_size)
    state_outdoor = step_fn_outdoor(state_outdoor, action)
    rollout_states_outdoor.append(state_outdoor)
    if state_outdoor.done > 0.5:
        break

print(f"Outdoor rollout: {len(rollout_states_outdoor)} steps")

In [ ]:
# Render outdoor vision frames
@jax.jit
def render_from_data_outdoor(data):
    """Render from outdoor env mjx.Data, stripping batch dim."""
    batched_data = add_batch_dim(data)
    result = vision_renderer_outdoor.render(batched_data)
    if isinstance(result, tuple):
        return result[0][0], result[1][0]
    return result[0]


n_outdoor_vision = min(500, len(rollout_states_outdoor))
vision_images_outdoor = []
for state_od in tqdm(
    rollout_states_outdoor[:n_outdoor_vision:render_every], desc="Outdoor vision render"
):
    result = render_from_data_outdoor(state_od.data)
    if isinstance(result, tuple):
        vision_images_outdoor.append(np.array(result[0]))
    else:
        vision_images_outdoor.append(np.array(result))

print(f"Collected {len(vision_images_outdoor)} outdoor vision frames")
print(f"Frame shape: {vision_images_outdoor[0].shape}")
print(
    f"Value range: [{vision_images_outdoor[0].min():.3f}, {vision_images_outdoor[0].max():.3f}]"
)

In [ ]:
# Side-by-side comparison: Default vs Outdoor Natural egocentric vision
n_compare = min(len(vision_images), len(vision_images_outdoor))
sample_idx = np.linspace(0, n_compare - 1, 8, dtype=int)

fig, axes = plt.subplots(2, len(sample_idx), figsize=(2.5 * len(sample_idx), 5))
for col, idx in enumerate(sample_idx):
    axes[0, col].imshow(vision_images[idx])
    axes[0, col].set_title(f"Step {idx * render_every}", fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(vision_images_outdoor[idx])
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Default\n(grey/dark)", fontsize=11)
axes[1, 0].set_ylabel("Outdoor Natural\n(grass/sky)", fontsize=11)
plt.suptitle(
    f"Egocentric Vision Comparison ({WIDTH}x{HEIGHT}) — Warp GPU Renderer",
    fontsize=13,
)
plt.tight_layout()
plt.show()

# Also show as side-by-side video
outdoor_uint8 = [
    np.clip(f * 255, 0, 255).astype(np.uint8) for f in vision_images_outdoor
]
n_vid = min(len(vision_uint8), len(outdoor_uint8))
# Concatenate default (left) and outdoor (right) with a white separator
sep = np.ones((HEIGHT, 4, 3), dtype=np.uint8) * 200
sidebyside = [
    np.concatenate([vision_uint8[i], sep, outdoor_uint8[i]], axis=1)
    for i in range(n_vid)
]
media.show_video(
    sidebyside,
    fps=fps // render_every,
    title="Egocentric Vision: Default (left) vs Outdoor Natural (right)",
)

## 5. Concatenated Video: Scene + Warp Vision Overlay

The JAX-native egocentric vision output is overlaid on the upper-left corner of the
main scene render.

In [ ]:
def overlay_vision_on_frame(scene_frame, vision_frame, scale=3, padding=8, border=2):
    """Overlay a small vision frame on the upper-left corner of a scene frame.

    Args:
        scene_frame: (H, W, 3) uint8 main camera frame.
        vision_frame: (h, w, 3) float32 [0,1] or uint8 egocentric vision frame.
        scale: Upscale factor for the vision inset.
        padding: Pixel padding from the top-left corner.
        border: Border width around the vision inset.

    Returns:
        (H, W, 3) uint8 frame with vision overlay.
    """
    frame = scene_frame.copy()

    # Convert float32 [0,1] to uint8 if needed
    if vision_frame.dtype != np.uint8:
        vision_frame = np.clip(vision_frame * 255, 0, 255).astype(np.uint8)

    vh, vw = vision_frame.shape[:2]
    sh, sw = vh * scale, vw * scale

    # Upscale vision frame using nearest-neighbor
    vision_up = np.kron(vision_frame, np.ones((scale, scale, 1))).astype(np.uint8)

    # Draw border (dark background)
    y0 = padding
    x0 = padding
    frame[
        y0 - border : y0 + sh + border,
        x0 - border : x0 + sw + border,
    ] = 32  # dark gray border

    # Paste vision
    frame[y0 : y0 + sh, x0 : x0 + sw] = vision_up

    return frame

In [ ]:
# Build concatenated frames
n_concat = min(len(frames_profile), len(vision_images))
concat_frames = []
for i in range(n_concat):
    concat_frames.append(
        overlay_vision_on_frame(frames_profile[i], vision_images[i], scale=3)
    )

print(f"Concatenated {n_concat} frames")
media.show_video(
    concat_frames,
    fps=fps // render_every,
    title="Scene + JAX-native Egocentric Vision Overlay",
)

In [ ]:
# Save all videos to disk (use the notebook's own directory)
output_dir = os.path.dirname(os.path.abspath("__file__"))

media.write_video(
    os.path.join(output_dir, "run_gap_close_profile.mp4"),
    frames_profile,
    fps=fps // render_every,
)
media.write_video(
    os.path.join(output_dir, "run_gap_egocentric_vision.mp4"),
    vision_uint8,
    fps=fps // render_every,
)
media.write_video(
    os.path.join(output_dir, "run_gap_scene_with_vision.mp4"),
    concat_frames,
    fps=fps // render_every,
)

print(f"Saved videos to {output_dir}/")